In [ ]:
from pathlib import Path
import pandas as pd

project_dir = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "code" / "notebooks").is_dir()
    ),
    Path.cwd(),
)
data_dir = project_dir / "data" / "raw" / "m5-forecasting-accuracy"

print("Dataset folder found:", data_dir.exists(), data_dir)

for filename in [
    "calendar.csv",
    "sell_prices.csv",
    "sales_train_validation.csv",
]:
    sample = pd.read_csv(data_dir / filename, nrows=5)
    print(f"\n{filename}: preview of {sample.shape[0]} rows")
    print(sample.iloc[:2, :8].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

sales = pd.read_csv(data_dir / "sales_train_validation.csv", nrows=1)
calendar = pd.read_csv(data_dir / "calendar.csv")

day_cols = [col for col in sales.columns if col.startswith("d_")]
dates = calendar.set_index("d").loc[day_cols, "date"]

series = pd.DataFrame({
    "date": pd.to_datetime(dates.to_numpy()),
    "units_sold": sales.loc[0, day_cols].astype(int).to_numpy(),
})

print("Item:", sales.loc[0, "item_id"])
print("Store:", sales.loc[0, "store_id"])
print("Date range:", series["date"].min().date(), "to", series["date"].max().date())

series["units_sold"].plot(figsize=(12, 4), alpha=0.5, label="Daily sales")
series["units_sold"].rolling(28).mean().plot(label="28-day average")
plt.title("Daily sales for one item at one store")
plt.xlabel("Date")
plt.ylabel("Units sold")
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(series["date"], series["units_sold"], alpha=0.45, label="Daily sales")
ax.plot(
    series["date"],
    series["units_sold"].rolling(28).mean(),
    label="28-day average",
)

ax.set(title="Daily sales for one item at one store", xlabel="Date", ylabel="Units sold")
ax.legend()
fig.autofmt_xdate()
plt.show()

In [ ]:
horizon = 28

train = series.iloc[:-horizon]
validation = series.iloc[-horizon:]

print("Training days:", len(train))
print("Validation days:", len(validation))
print("Training ends:", train["date"].max().date())
print("Validation starts:", validation["date"].min().date())

In [ ]:
recent_average = train["units_sold"].tail(horizon).mean()

validation_with_forecast = validation.copy()
validation_with_forecast["forecast_units"] = recent_average

print("Average daily sales in the last 28 training days:", round(recent_average, 3))
print(validation_with_forecast[["date", "units_sold", "forecast_units"]].head())

In [ ]:
# mean absolute error (MAE)

absolute_errors = (
    validation_with_forecast["units_sold"]
    - validation_with_forecast["forecast_units"]
).abs()

mae = absolute_errors.mean()

print("MAE:", round(mae, 3))
print("First five daily errors:", absolute_errors.head().tolist())

In [ ]:
train.tail(7)

In [ ]:
train["units_sold"].tail(7)

In [ ]:
weekly_pattern = train["units_sold"].tail(7).tolist()
weekly_pattern

In [ ]:
weekly_forecast = weekly_pattern * 4
len(weekly_forecast)

In [ ]:
validation_with_weekly = validation.copy()
validation_with_weekly["weekly_forecast"] = weekly_forecast
validation_with_weekly.head(7)

In [ ]:
validation_with_weekly["weekly_error"] = (validation_with_weekly["units_sold"] - validation_with_weekly["weekly_forecast"]).abs()

In [ ]:
validation_with_weekly["weekly_error"].mean()

In [ ]:
avg_last_7_days = train['units_sold'].tail(7).mean()
avg_last_7_days

In [ ]:
validation_with_7day = validation.copy()
validation_with_7day['forecast_7day_avg'] = avg_last_7_days
validation_with_7day.head()

In [ ]:
validation_with_7day['error_7day_avg'] = (validation_with_7day['units_sold'] - validation_with_7day['forecast_7day_avg']).abs()

In [ ]:
validation_with_7day['error_7day_avg'].mean()

Checking 28-days model for different product and store......

In [ ]:
sample_sales = pd.read_csv(data_dir / "sales_train_validation.csv", nrows=100)
sample_sales.shape

In [ ]:
sample_sales.head()

In [ ]:
sample_sales.iloc[:2,:10]

In [ ]:
daily_sales = sample_sales.iloc[:,6:]
daily_sales

In [ ]:
sample_train_sales = sample_sales.iloc[: , 6:-28]
sample_validation_sales = sample_sales.iloc[: , -28:]
sample_train_sales.shape,sample_validation_sales.shape

In [ ]:
sample_train_sales

In [ ]:
sample_validation_sales

In [ ]:
last_7_day = sample_train_sales.iloc[:,-7:]
avg_last_7_day = last_7_day.mean(axis=1)
avg_last_7_day.head()

In [ ]:
forecast_7day = pd.DataFrame(
    index=sample_validation_sales.index,
    columns=sample_validation_sales.columns
)

for day in forecast_7day.columns:
    forecast_7day[day] = avg_last_7_day

forecast_7day.iloc[:5, :5]

In [ ]:
errors_7day = (sample_validation_sales - forecast_7day).abs()
errors_7day.iloc[:5, :5]

In [ ]:
mae_7day = errors_7day.mean().mean()
mae_7day

Calculating mae for 28-Days

In [ ]:
last_28_day = sample_train_sales.iloc[:,-28:]
avg_last_28_day = last_28_day.mean(axis=1)
avg_last_28_day.head()

In [ ]:
forecast_28day = pd.DataFrame(
    index=sample_validation_sales.index,
    columns=sample_validation_sales.columns
)

for day in forecast_28day.columns:
    forecast_28day[day] = avg_last_28_day

forecast_28day.iloc[:5, :5]

In [ ]:
avg_last_28_day = sample_train_sales.iloc[:, -28:].mean(axis=1)
avg_last_28_day.head()

In [ ]:
avg_last_28_day.head()

In [ ]:
errors_28day = (sample_validation_sales - forecast_28day).abs()
errors_28day.iloc[:5, :5]

In [ ]:
mae_28day = errors_28day.mean().mean()
mae_28day

Calculating last week sales till 28 days

In [ ]:
last_week_sales = sample_train_sales.iloc[:, -7:]
last_week_sales.iloc[:5, :]

In [ ]:
forecast_weekly = pd.concat([last_week_sales] * 4, axis=1)
forecast_weekly.shape

In [ ]:
forecast_weekly.columns = sample_validation_sales.columns
forecast_weekly.iloc[:5, :10]

In [ ]:
errors_weekly =(sample_validation_sales - forecast_weekly).abs()
errors_weekly.iloc[:5 , :10]

In [ ]:
mae_weekly = errors_weekly.mean().mean()
mae_weekly

Calculating for 1,000 series sampled from across the full dataset.

In [ ]:
all_sales = pd.read_csv(data_dir / "sales_train_validation.csv")
all_sales.shape

In [ ]:
sample_1000 = all_sales.sample(n=1000, random_state=50)
sample_1000.shape

In [ ]:
sample_1000

In [ ]:
daily_sales_1000 = sample_1000.iloc[: , 6:]
daily_sales_1000.shape

In [ ]:
daily_sales_1000

In [ ]:
train_sales_1000 = daily_sales_1000.iloc[:, :-28]
validation_sales_1000 = daily_sales_1000.iloc[:, -28:]
train_sales_1000.shape, validation_sales_1000.shape

In [ ]:
avg_last_28_1000 = train_sales_1000.iloc[:, -28:].mean(axis=1)
avg_last_28_1000.head()

In [ ]:
forecast_28_1000 = pd.DataFrame(
    index = validation_sales_1000.index,
    columns= validation_sales_1000.columns
)

for day in forecast_28_1000.columns:
    forecast_28_1000[day] = avg_last_28_1000

forecast_28_1000.iloc[:5,:5]

In [ ]:
error_28_1000 = (validation_sales_1000 - forecast_28_1000).abs()
error_28_1000.iloc[:5 , :5]

In [ ]:
mea_28_1000 = error_28_1000.mean().mean()
mea_28_1000

for last 7 days

In [ ]:
avg_last_7_1000 = train_sales_1000.iloc[:, -7:].mean(axis=1)
avg_last_7_1000.head()

In [ ]:
forecast_7avg_1000 = pd.DataFrame(
    index=validation_sales_1000.index,
    columns=validation_sales_1000.columns
)

for day in forecast_7avg_1000.columns:
    forecast_7avg_1000[day] = avg_last_7_1000

forecast_7avg_1000.iloc[:5 , :5]

In [ ]:
error_7avg_1000 = (validation_sales_1000 - forecast_7avg_1000).abs()
error_7avg_1000.iloc[:5 , :5]

In [ ]:
mea_7avg_1000 = error_7avg_1000.mean().mean()
mea_7avg_1000

Calculating for last_week

In [ ]:
last_week_1000 = train_sales_1000.iloc[:, -7:]
last_week_1000.iloc[:5, :]

In [ ]:
forecast_weekly_1000 = pd.concat([last_week_1000] * 4, axis=1)
forecast_weekly_1000.shape

In [ ]:
forecast_weekly_1000.columns = validation_sales_1000.columns
forecast_weekly_1000.iloc[:5 , :10]

In [ ]:
error_weekly_1000 = (validation_sales_1000 - forecast_weekly_1000).abs()
error_weekly_1000.iloc[:5,:5]

In [ ]:
mae_weekly_1000 = error_weekly_1000.mean().mean()
mae_weekly_1000

Testing for all 30490 product-store series.

In [ ]:
daily_sales_all = all_sales.iloc[:, 6:]
train_sales_all = daily_sales_all.iloc[:, :-28]
validation_sales_all = daily_sales_all.iloc[:, -28:]
train_sales_all.shape, validation_sales_all.shape

In [ ]:
avg_last_28_all = train_sales_all.iloc[:, -28:].mean(axis=1)
avg_last_28_all.head()

In [ ]:
forecast_28_all = pd.DataFrame(
    index = validation_sales_all.index,
    columns = validation_sales_all.columns
)

for day in forecast_28_all.columns:
    forecast_28_all[day] = avg_last_28_all

forecast_28_all.iloc[:5 , :5]

In [ ]:
error_28_all = (validation_sales_all - forecast_28_all).abs()
error_28_all.iloc[:5 , :5]

In [ ]:
mae_28_all = error_28_all.mean().mean()
mae_28_all

Calculating mae for last 7-days

In [ ]:
avg_last_7_all = train_sales_all.iloc[: , -7:].mean(axis=1)
avg_last_7_all.head()

In [ ]:
forecast_7avg_all = pd.DataFrame(
    index = validation_sales_all.index,
    columns = validation_sales_all.columns
)

for day in forecast_7avg_all.columns:
    forecast_7avg_all[day] = avg_last_7_all

forecast_7avg_all.iloc[:5 , :5]

In [ ]:
error_7avg_all = (validation_sales_all - forecast_7avg_all).abs()
error_7avg_all.iloc[:5 , :5]

In [ ]:
mae_7avg_all = error_7avg_all.mean().mean()
mae_7avg_all

Calculating for last_week

In [ ]:
last_week_all = train_sales_all.iloc[: , -7:]
last_week_all.iloc[:5 , :]

In [ ]:
forecast_weekly_all = pd.concat([last_week_all] * 4 , axis=1)
forecast_weekly_all.shape

In [ ]:
forecast_weekly_all.columns = validation_sales_all.columns
forecast_weekly_all.head()

In [ ]:
error_weekly_all = (validation_sales_all - forecast_weekly_all).abs()
error_weekly_all.iloc[:5 , :5]

In [ ]:
mae_weekly_all = error_weekly_all.mean().mean()
mae_weekly_all

Result of all MAE

In [ ]:
baseline_result_all = pd.DataFrame({
    "baseline" : ["28-day avg","7-day avg","repeat_last_week"],
    "MAE" : [mae_28_all,mae_7avg_all,mae_weekly_all]
}).sort_values("MAE")

baseline_result_all

time-series backtesting from [:, :-56]

In [ ]:
train_window_1 = daily_sales_all.iloc[: , :-56]
validation_window_1 = daily_sales_all.iloc[: , -56 : -28]
train_window_1.shape , validation_window_1.shape

last_28-days testing.

In [ ]:
avg_last_28_window_1 = train_window_1.iloc[: , -28:].mean(axis=1)
avg_last_28_window_1.head()

In [ ]:
forecast_28_window_1 = pd.DataFrame(
    index = validation_window_1.index,
    columns = validation_window_1.columns
)

for day in forecast_28_window_1.columns:
    forecast_28_window_1[day] = avg_last_28_window_1

forecast_28_window_1.iloc[:5,:5]

In [ ]:
error_28_window_1 = (validation_window_1 - forecast_28_window_1).abs()
error_28_1000.iloc[:5,:5]

In [ ]:
mae_28_window_1 = error_28_window_1.mean().mean()
mae_28_window_1

calculating mae for last 7-days

In [ ]:
avg_last_7_window_1 = train_window_1.iloc[: , -7:].mean(axis=1)
avg_last_7_window_1.head()

In [ ]:
forecast_7avg_window_1 = pd.DataFrame(
    index = validation_window_1.index,
    columns = validation_window_1.columns
)

for day in forecast_7avg_window_1.columns:
    forecast_7avg_window_1[day] = avg_last_7_window_1

forecast_7avg_window_1.iloc[:5,:5]

In [ ]:
error_7avg_window_1 = (validation_window_1 - forecast_7avg_window_1).abs()
error_7avg_window_1.iloc[:5,:5]

In [ ]:
mae_7avg_window_1 = error_7avg_window_1.mean().mean()
mae_7avg_window_1

calculating for last_week_window_1

In [ ]:
last_week_window_1 = train_window_1.iloc[: , -7:]
last_week_window_1.iloc[:5,:]

In [ ]:
forecast_weekly_window_1 = pd.concat([last_week_window_1] * 4 , axis=1)
forecast_weekly_window_1.shape

In [ ]:
forecast_weekly_window_1.columns = validation_window_1.columns

In [ ]:
error_weekly_window_1 = (validation_window_1 - forecast_weekly_window_1).abs()
error_weekly_window_1.iloc[:5,:5]

In [ ]:
mae_weekly_window_1 = error_weekly_window_1.mean().mean()
mae_weekly_window_1

Result of all mae_window_1

In [ ]:
baseline_result_window_1 = pd.DataFrame({
    "baseline" : ["28_avg" , "7avg", "repeat_last_week"],
    "MAE" : [mae_28_window_1,mae_7avg_window_1,mae_weekly_window_1]
}).sort_values("MAE")

baseline_result_window_1

2nd time-series backtesting from [:, :-84]

In [ ]:
train_window_2 = daily_sales_all.iloc[:, :-84]
validation_window_2 = daily_sales_all.iloc[:, -84:-56]

train_window_2.shape, validation_window_2.shape

In [ ]:
avg_last_28_window_2 = train_window_2.iloc[: , -28:].mean(axis=1)
avg_last_28_window_2.head()

In [ ]:
forecast_28_window_2 = pd.DataFrame(
    index = validation_window_2.index,
    columns = validation_window_2.columns
)

for day in forecast_28_window_2.columns:
    forecast_28_window_2[day] = avg_last_28_window_2

forecast_28_window_2.iloc[:5,:5]

In [ ]:
error_28_window_2 = (validation_window_2 - forecast_28_window_2).abs()
error_28_window_2.iloc[:5,:5]

In [ ]:
mae_28_window_2 = error_28_window_2.mean().mean()
mae_28_window_2

In [ ]:
avg_last_7_window_2 = train_window_1.iloc[: , -7:].mean(axis=1)
avg_last_7_window_2.head()

In [ ]:
forecast_7avg_window_2 = pd.DataFrame(
    index= validation_window_2.index,
    columns= validation_window_2.columns
)

for day in forecast_7avg_window_2.columns:
    forecast_7avg_window_2[day] = avg_last_7_window_2

forecast_7avg_window_2.iloc[:5,:5]

In [ ]:
error_7avg_window_2 = (validation_window_2 - forecast_7avg_window_2).abs()
error_7avg_window_2.iloc[:5,:5]

In [ ]:
mae_7avg_window_2 = error_7avg_window_2.mean().mean()
mae_7avg_window_2

In [ ]:
last_week_window_2 = train_window_2.iloc[: , -7:]
last_week_window_2.iloc[:5 , :]

In [ ]:
forecast_weekly_window_2 = pd.concat([last_week_window_2] * 4 , axis=1)
forecast_weekly_window_2.shape

In [ ]:
forecast_weekly_window_2.columns = validation_window_2.columns

In [ ]:
error_weekly_window_2 = (validation_window_2 - forecast_weekly_window_2).abs()
error_weekly_window_2.iloc[:5,:5]

In [ ]:
mae_weekly_window_2 = error_weekly_window_2.mean().mean()
mae_weekly_window_2

In [ ]:
baseline_result_window_2 = pd.DataFrame({
    "baseline" : ("28_days_mae","7_days_mae","weekly_mae"),
    "MAE" : (mae_28_window_2,mae_7avg_window_2,mae_weekly_window_2)
}).sort_values("MAE")
baseline_result_window_2

In [ ]:
baseline_result_summary = pd.DataFrame({
    "baseline" : ("28_days_mae","7_days_mae","weekly_mae"),
    "MAE_latest_window" : (mae_28_all,mae_7avg_all,mae_weekly_all),
    "MAE_window_1" : (mae_28_window_1,mae_7avg_window_1,mae_weekly_window_1),
    "MAE_window_2" : (mae_28_window_2,mae_7avg_window_2,mae_weekly_window_2)
})

baseline_result_summary["Mean_MAE"] = baseline_result_summary[
    ["MAE_latest_window","MAE_window_1","MAE_window_2"]
].mean(axis=1)

baseline_result_summary.sort_values("Mean_MAE")

In [ ]:
last_28_window_1 = train_window_1.iloc[:, -28:]

weekday_average_window_1 = pd.DataFrame(index=train_window_1.index)

for weekday_position in range(7):
    weekday_average_window_1[weekday_position] = (
        last_28_window_1.iloc[:, weekday_position::7].mean(axis=1)
    )

weekday_average_window_1.head()

In [ ]:
forecast_weekday_window_1 = pd.concat(
    [weekday_average_window_1] * 4,
    axis=1
)

forecast_weekday_window_1.columns = validation_window_1.columns

forecast_weekday_window_1.iloc[:5, :10]

In [ ]:
error_weekday_window_1 = (
    validation_window_1 - forecast_weekday_window_1
).abs()

error_weekday_window_1.iloc[:5, :10]

In [ ]:
mae_weekday_window_1 = error_weekday_window_1.mean().mean()
mae_weekday_window_1

Checking last_28_day for window_2...

In [ ]:
last_28_window_2 = train_window_2.iloc[:, -28:]

weekday_average_window_2 = pd.DataFrame(index=train_window_2.index)

for weekday_position in range(7):
    weekday_average_window_2[weekday_position] = (
        last_28_window_2.iloc[:, weekday_position::7].mean(axis=1)
    )

weekday_average_window_2.head()

In [ ]:
forecast_weekday_window_2 = pd.concat(
    [weekday_average_window_2] * 4,
    axis=1
)

forecast_weekday_window_2.columns = validation_window_2.columns

forecast_weekday_window_2.iloc[:5, :10]

In [ ]:
error_weekday_window_2 = (
    validation_window_2 - forecast_weekday_window_2
).abs()

error_weekday_window_2.iloc[:5, :10]

In [ ]:
mae_weekday_window_2 = error_weekday_window_2.mean().mean()
mae_weekday_window_2

In [ ]:
last_28_latest = train_sales_all.iloc[:, -28:]

weekday_average_latest = pd.DataFrame(index=train_sales_all.index)

for weekday_position in range(7):
    weekday_average_latest[weekday_position] = (
        last_28_latest.iloc[:, weekday_position::7].mean(axis=1)
    )

weekday_average_latest.head()

In [ ]:
forecast_weekday_latest = pd.concat(
    [weekday_average_latest] * 4,
    axis=1
)

forecast_weekday_latest.columns = validation_sales_all.columns

forecast_weekday_latest.iloc[:5, :10]

In [ ]:
error_weekday_latest = (
    validation_sales_all - forecast_weekday_latest
).abs()

error_weekday_latest.iloc[:5, :10]

In [ ]:
mae_weekday_latest = error_weekday_latest.mean().mean()
mae_weekday_latest

In [ ]:
baseline_result_summary = pd.DataFrame({
    "baseline": [
        "28-day avg",
        "7-day avg",
        "repeat last week",
        "weekday-aware avg"
    ],
    "MAE_latest": [
        mae_28_all, mae_7avg_all, mae_weekly_all, mae_weekday_latest
    ],
    "MAE_window_1": [
        mae_28_window_1, mae_7avg_window_1,
        mae_weekly_window_1, mae_weekday_window_1
    ],
    "MAE_window_2": [
        mae_28_window_2, mae_7avg_window_2,
        mae_weekly_window_2, mae_weekday_window_2
    ]
})

baseline_result_summary["Mean_MAE"] = baseline_result_summary[
    ["MAE_latest", "MAE_window_1", "MAE_window_2"]
].mean(axis=1)

baseline_result_summary.sort_values("Mean_MAE")